# SGLang 本地单卡入门部署 — 快速跑通大模型推理（入门实战）

**定位**：用 [SGLang](https://github.com/sgl-project/sglang) 在 **单张 NVIDIA GPU** 上拉起 **OpenAI 兼容 HTTP 服务**，适合第一次接触 SGLang 推理部署的同学；不涉及框架源码修改、RadixAttention 调优、多机多卡与高阶性能工程。

**核心口诀**：装好 PyTorch + SGLang 后，终端里 **一条命令** 即可启动服务：

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 适用显卡与显存（新手照表选模型）

| 项目 | 说明 |
|------|------|
| **显卡** | **NVIDIA GPU**（推荐 Turing / Ampere / Ada / Blackwell 等较新架构）；本 Demo 按 **单卡** 编写。无 NVIDIA GPU 时无法按本文运行 SGLang CUDA 路径。 |
| **显存** | 默认模型 `Qwen/Qwen2.5-0.5B-Instruct`：**建议 ≥ 6GB 显存**（宽松建议 **≥ 8GB** 更稳）。若改 **7B** 级别模型，常见需要 **≥ 16GB**。 |
| **驱动** | 安装与所用 **PyTorch / SGLang 轮子**匹配的 NVIDIA 驱动；驱动过旧会导致 `CUDA driver version is insufficient` 类错误。 |
| **CUDA** | 使用与 PyTorch 官方索引一致的 CUDA 运行时版本（如 **cu121 / cu124**）；**勿**混装多个冲突的 CUDA 工具包到同一虚拟环境。 |
| **Python** | 代码语法兼容 **Python 3.9 – 3.12**；具体版本下限以 `pip install sglang` 时的报错或 [官方安装文档](https://sgl-project.github.io/) 为准。 |
| **系统** | **Linux x86_64** 最省心；Windows 用户建议在 WSL2 或独立终端运行服务，Notebook 内客户端代码同样适用。 |

---

## 完整依赖安装命令（终端执行，推荐先建虚拟环境）

```bash
# 0) 进入你的工作目录，创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate
# Windows（cmd/PowerShell）: .venv\Scripts\activate

# 1) 升级 pip，减少解析依赖时的怪异报错
python -m pip install --upgrade pip

# 2) 安装带 CUDA 的 PyTorch（下面示例为 CUDA 12.4 轮子索引）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3) 安装 SGLang（全部组件）与 OpenAI 官方 Python SDK
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"

# 4) 运行本 Notebook 需要 Jupyter（任选其一）
pip install notebook ipykernel
# 或: pip install jupyterlab
```

> **提示**：`sglang` 与 `torch` 的版本组合以当时 PyPI 解析结果为准；若安装失败，请根据终端报错调整 Python 版本或 CUDA 索引，并查阅 SGLang 官方文档。

---

## 简单使用说明

1. 克隆或下载包含本文件的仓库，进入目录，按上一节 **完成依赖安装** 并激活虚拟环境。
2. 启动 Jupyter：`jupyter notebook`，打开本 Notebook。
3. **方式 A（终端一行启动）**：在系统终端执行上文 **核心口诀** 中的命令，待日志显示服务就绪后，在 Notebook 里只运行 **环境自检** 与 **调用 API** 单元格。
4. **方式 B（全在 Notebook 内）**：从上到下依次执行：环境自检 →（可选）pip 安装 → 启动服务 → 调用 API → 用完后 **停止服务** 释放显存。
5. 首次运行会从 Hugging Face 拉取模型，需网络可达或已配置镜像/缓存；下载时间与带宽有关。

---

In [ ]:
# === 环境快速自检 ===

import sys  # 导入标准库 sys，用于读取当前解释器版本信息

print("Python 版本:", sys.version)  # 打印 Python 版本字符串，确认与 SGLang 要求一致

try:
    import torch  # 尝试导入 PyTorch，若未安装会进入 except 分支
except ImportError:
    print("未检测到 torch：请先安装带 CUDA 的 PyTorch（见上文 pip 命令）。")  # 给出下一步指引
else:
    print("torch 版本:", torch.__version__)  # 显示已安装的 torch 版本号
    print("CUDA 是否可用:", torch.cuda.is_available())  # True 表示当前进程能看到 GPU
    if torch.cuda.is_available():
        print("GPU0 名称:", torch.cuda.get_device_name(0))  # 打印第一块 GPU 的产品名
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)  # 计算总显存 GB
        print("GPU0 显存(GB, 约):", vram_gb)  # 打印总显存便于对照上文建议

In [ ]:
# 可选：在 Notebook 当前内核环境中安装 SGLang 与客户端依赖（若终端已装好可跳过）

import subprocess  # 导入 subprocess，用子进程调用 pip
import sys  # 导入 sys，拿到当前 Jupyter 内核对应的 python 可执行路径

_packages = ["sglang[all]", "openai>=1.0.0", "requests>=2.28.0"]  # 固定要装的包及最低版本约束
_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "pip"] + _packages  # 组装 pip install 命令
subprocess.run(_cmd, check=True)  # 运行安装，失败则抛异常便于立刻看到 pip 报错
print("依赖安装命令已执行完毕；若缺少 torch/cuda，请仍按文档单独安装 PyTorch。")  # 提示 PyTorch 需与显卡匹配

## 在 Notebook 内启动 SGLang 推理服务

下面的代码会在 Notebook 里 **启动后台子进程** 运行 SGLang 推理服务。你也可以 **复制打印出来的命令** 到终端单独运行，更符合「一条命令启动」的习惯。

---

In [ ]:
# 在 Notebook 内启动 SGLang OpenAI 兼容推理服务
# 若你已在终端手动执行命令，请跳过本格，直接运行后面的「调用 API」

import json  # 用于解析 HTTP 返回的 JSON
import subprocess  # 用于创建后台子进程
import sys  # 当前 Python 解释器路径
import time  # 用于轮询等待服务就绪
import urllib.error  # 捕获网络与 HTTP 异常
import urllib.request  # 标准库发 HTTP GET，无需额外安装

HOST = "127.0.0.1"  # 监听地址：仅本机访问，降低误暴露风险
PORT = 30000  # 监听端口：SGLang 默认使用 30000
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Hugging Face 模型 ID，小显存友好

_sglang_process = None  # 全局子进程句柄占位，启动成功后变为 Popen 实例


def build_sglang_command():
    """构建 SGLang 启动命令列表，并打印可复制的终端一行版。"""
    parts = [  # 组装启动命令参数列表
        sys.executable, "-m", "sglang.launch_server",  # 使用当前 Python 解释器以模块方式启动
        "--model-path", MODEL_ID,  # 指定模型路径（Hugging Face ID 或本地路径）
        "--host", HOST,  # 绑定监听地址
        "--port", str(PORT),  # 绑定监听端口
    ]
    shell_one_liner = " ".join(parts)  # 拼成一条字符串，便于复制到终端
    print("终端一行启动（可直接复制）:")  # 提示用户
    print(shell_one_liner)  # 打印完整命令
    return parts  # 返回列表供 Popen 使用


def start_sglang_if_not_running():
    """启动子进程并轮询 /v1/models 直到可用或超时。"""
    global _sglang_process  # 声明修改模块级变量以保存进程句柄
    if _sglang_process is not None and _sglang_process.poll() is None:
        print("检测到 SGLang 已在运行，跳过重复启动。")  # 避免多实例占满显存
        return
    cmd = build_sglang_command()  # 生成命令并打印一行版
    _sglang_process = subprocess.Popen(  # 以子进程方式后台启动 SGLang
        cmd,
        stdout=subprocess.DEVNULL,  # 丢弃标准输出避免管道阻塞
        stderr=subprocess.STDOUT,  # 合并错误输出到标准输出（同样丢弃）
        start_new_session=True,  # 创建新会话组，便于后续整组 kill
    )
    base = f"http://{HOST}:{PORT}"  # 服务根 URL
    models_url = f"{base}/v1/models"  # OpenAI 兼容模型列表端点，用于探活
    deadline = time.time() + 600  # 最长等待 600 秒，给首次下载模型留时间
    print("正在等待 SGLang 服务就绪（首次需下载模型，可能耗时较长）...")  # 告知用户正在等待
    while time.time() < deadline:  # 在截止时间前持续轮询
        if _sglang_process.poll() is not None:  # 检查子进程是否已退出
            raise RuntimeError(  # 子进程意外退出，立刻报错
                "SGLang 进程已退出，退出码 %s。请在终端手动运行上面的命令查看完整日志。" % _sglang_process.returncode
            )
        try:
            with urllib.request.urlopen(models_url, timeout=2) as resp:  # 发送 GET 请求探活
                body = resp.read().decode("utf-8")  # 读取响应文本
            data = json.loads(body)  # 解析为字典
            print("服务已就绪！/v1/models 返回键:", list(data.keys()))  # 确认接口可用
            print("OpenAI 兼容 Base URL:", f"http://{HOST}:{PORT}/v1")  # 客户端填写的根路径
            return  # 服务就绪，退出等待循环
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ConnectionResetError, json.JSONDecodeError):
            time.sleep(2)  # 未就绪则等待 2 秒再试
    raise TimeoutError("等待 SGLang 就绪超时（600s），请检查网络、显存与日志。")  # 超时报错


start_sglang_if_not_running()  # 执行启动流程

## 调用本地 SGLang 推理服务

SGLang 提供 OpenAI 兼容的 `/v1/chat/completions` 接口，可直接使用 `openai` Python SDK 调用，与调用 OpenAI 官方 API 的代码几乎一致。

---

In [ ]:
# 调用本地 SGLang 提供的 OpenAI 兼容 Chat Completions

from openai import OpenAI  # 导入 OpenAI 官方 SDK 的客户端类（1.x 风格）

client = OpenAI(  # 创建 OpenAI 客户端实例
    base_url="http://127.0.0.1:30000/v1",  # 指向本地 SGLang 服务的 /v1 端点
    api_key="EMPTY",  # 本地服务不校验密钥，使用占位字符串
)

completion = client.chat.completions.create(  # 发起非流式聊天补全请求
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称，须与启动时一致
    messages=[  # 构造对话消息列表
        {"role": "system", "content": "你是一个帮助新手的简短中文助手。"},  # 系统提示词，设定角色
        {"role": "user", "content": "用一句话说明 SGLang 是做什么的。"},  # 用户问题
    ],
    temperature=0.2,  # 较低温度使输出更确定
    max_tokens=128,  # 限制最大生成 token 数
)

text = completion.choices[0].message.content  # 取出第一条回复的正文
print("模型回复:", text)  # 打印模型输出
print("token 用量:", completion.usage)  # 打印 prompt/completion token 统计

## 停止服务释放显存

实验完成后请务必运行下方单元格，停止 SGLang 子进程并释放 GPU 显存。

---

In [ ]:
# 停止 Notebook 内启动的 SGLang 子进程，释放 GPU 显存

import os  # 导入 os，用于向进程组发送信号
import signal  # 导入 signal 模块，使用 SIGTERM 优雅退出

try:
    _sglang_process  # 检查变量是否存在，若未运行过启动格会 NameError
except NameError:
    print("未找到 _sglang_process：若服务在终端启动，请在终端 Ctrl+C 结束。")  # 提示用户手动停服务
else:
    if _sglang_process is None or _sglang_process.poll() is not None:  # 进程为空或已退出
        print("没有正在运行的 SGLang 子进程。")  # 无需处理
    else:
        try:
            os.killpg(os.getpgid(_sglang_process.pid), signal.SIGTERM)  # 对整个进程组发 SIGTERM
        except ProcessLookupError:
            _sglang_process.terminate()  # 进程组接口失败则直接 terminate 子进程
        _sglang_process.wait(timeout=30)  # 等待最多 30 秒让进程释放资源
        print("已停止 SGLang 子进程，GPU 显存已释放。")  # 告知用户操作完成

## 常见报错与排查

### 1）`CUDA out of memory` / 显存不足

**原因**：当前模型加载或推理时超出 GPU 显存容量。

**解决**：
- 换用更小的模型（本 Demo 已使用 0.5B 级别）
- 关闭其它占用显存的程序
- 在启动命令中添加 `--mem-fraction-static 0.8` 参数，限制 SGLang 使用的显存比例
- 使用量化模型减少显存占用

### 2）`Address already in use` / 端口被占用

**原因**：30000 端口已被其他进程（或上次未正常退出的 SGLang）占用。

**解决**：
- 查找并结束占用进程：`lsof -i :30000` 然后 `kill <PID>`
- 或更换端口：启动命令中将 `--port 30000` 改为其他未占用端口（如 `--port 30001`）

---

**结语**：掌握 SGLang 单卡部署后，可继续学习多轮对话、批量推理、RadixAttention 前缀缓存等进阶主题。